# Link chains

> Two link chains ready to use: uncoded bits on one carrier, and 5G NR with Sionna PHY.

In [ ]:
#| default_exp radio.link.presets

In [ ]:
#| hide
from nbdev.showdoc import *

Two `LinkChain`s (see `radio.link.chain`), each its stages put together:

| | `UncodedChain` | `NRChain` |
|---|---|---|
| coding | none (`NoCoding`, `HardDecision`) | 5G NR: CRC + LDPC (`NRTBEncoder`, `NRTBDecoder`) |
| modulation | any `Modulation` (BPSK, `SionnaModulation`) | the QAM of each transmission's MCS |
| MCS | none | fixed, or link adaptation from the SINR (`NRLinkAdaptation`) |
| waveform | one carrier (`SingleCarrier`): a flat channel | `OFDM`: each symbol on its subcarrier |
| equalization, demapping | one tap; LLRs (max-log for Sionna's constellations) | one tap; APP LLRs |
| delivery | with its bit errors (`drop_corrupted`: an ideal CRC) | the CRC's |
| needs | numpy | Sionna, PyTorch |

Use `UncodedChain` to study what bit errors do to a message (codebook tokens, quantized latents), and `NRChain` for a
realistic 5G transmission, which agrees with the system level (`SionnaErrorModel`): the same MCS table and resource
allocation. Both are radio channels' receivers through `LinkLevelChannel(chain=...)` (see `radio.link.channel`).

In [ ]:
#| export
from __future__ import annotations

from comm_core.channel.awgn import ApplyChannel
from comm_core.coding.nr import MIN_LDPC_CODERATE, NRTBDecoder, NRTBEncoder
from comm_core.coding.uncoded import HardDecision, NoCoding
from comm_core.equalization.one_tap import OneTapEqualizer
from comm_core.modulation.constellation import BPSK, Modulation, SionnaModulation
from comm_core.modulation.mapping import Demapper, Mapper
from comm_core.radio.link.chain import LinkChain
from comm_core.radio.link_adaptation import NRLinkAdaptation, NRMCSTable
from comm_core.source.codec import Codec
from comm_core.waveform.ofdm import OFDM
from comm_core.waveform.single_carrier import SingleCarrier

## Uncoded

Every bit of each transmission: payload → bits → symbols → `h √P x + n`, with the interference added to the noise →
detected bits → payload, with its bit errors. The payloads of a step with the same shape go through the stages together, each with its own noise
(`batch=False`: one at a time, in the step's order).

In [ ]:
#| export
class UncodedChain(LinkChain):
    "Uncoded bits on one carrier: payload -> bits -> symbols -> h √P x + n -> detected bits -> payload (bit errors pass)."

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        modulation: Modulation | None = None, # Bits <-> symbols; None: `BPSK()`
        drop_corrupted: bool = False, # Lose the packets with bit errors (an ideal CRC) instead of delivering them
        batch: bool = True, # Payloads of the same shape through the stages together (False: one at a time)
    ):
        self.modulation = modulation if modulation is not None else BPSK()
        mapper = Mapper(self.modulation)
        super().__init__([NoCoding(), mapper, SingleCarrier(), ApplyChannel(), OneTapEqualizer(), Demapper(mapper),
                          HardDecision()], codec, batch=batch, drop_corrupted=drop_corrupted)

## 5G NR, with Sionna PHY

The link level of 5G NR with [Sionna PHY](https://nvlabs.github.io/sionna/phy/index.html): every bit of every
transmission, coded and decoded the way an NR receiver does it.

1. The payload becomes bits (a `Codec`, e.g. `IndexCodec` for codebook indices).
2. **Link adaptation:** the MCS of each transmission, and its allocation (`NRLinkAdaptation`).
3. **Coding:** the bits become an NR transport block, with a CRC, LDPC coding and rate matching (`NRTBEncoder`), at
   the MCS of the transmission.
4. **Modulation:** the MCS's QAM (`Mapper`).
5. **The channel:** the symbols go out on resource elements, filling the subcarriers in turn (`OFDM`). Each sees the
   channel of its subcarrier: `y = h_k √P x + n`, where the noise includes any interference.
6. **The receiver:** it equalizes each resource element (`OneTapEqualizer`, knowing `h_k`), computes soft bits
   (`Demapper`, APP LLRs), and decodes the LDPC code (`NRTBDecoder`).

The CRC decides delivery, as in a real receiver: a transport block that fails its CRC is lost.

`NRChain` is these stages, and `LinkLevelChannel(NRChain(...))` the radio channel whose receiver it is. It takes
the same fadings (e.g. `RayTracedPathsFading` with subcarriers), power control, interference (`interference=True`)
and protocols as any other. It needs Sionna 2 (PyTorch): `pip install 'comm-core[sionna]'`.

**The MCS:** a fixed `mcs_index`, or link adaptation (the default). Link adaptation picks the highest MCS that
Sionna SYS (`SionnaErrorModel`) predicts to meet `bler_target` at the transmission's SINR. The transmitter is assumed
to know the channel.

**The results** add `mcs`, `resource_elements`, `tb_size` (the NR transport block, padded to an allowed size) and
`bit_errors` to their metadata. `error_rate` is the bit error rate of the decoded payload.

**MCSs:** those with code rates below 1/5 (MCS 0–2 of table 1) are not available. Sionna's LDPC encoder
doesn't support them (NR reaches them by repetition), so link adaptation chooses from the others.

**Cost:** LDPC decoding is heavy, around 25 ms per 1000-bit message on a CPU, much less on a GPU. Use it to check the
system level (`SionnaErrorModel`), or where the bits themselves matter. For training, `SystemLevelChannel` with
`SionnaErrorModel` is the fast equivalent.

In [ ]:
#| export
class NRChain(LinkChain):
    "5G NR with Sionna PHY: an MCS per transmission, CRC + LDPC, QAM, OFDM, one-tap equalization, APP demapping, LDPC decoding."

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        mcs_index: int | None = None, # A fixed MCS; None: link adaptation
        bler_target: float = 0.1, # Link adaptation's target error rate
        mcs_table_index: int = 1, # 5G NR MCS table: 1 (up to 64-QAM) or 2 (up to 256-QAM)
        mcs_category: int = 0, # 0: PUSCH (uplink), 1: PDSCH (downlink)
        num_bp_iter: int = 20, # LDPC decoder iterations
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
    ):
        from comm_core.radio.system.phy_abstraction import SionnaErrorModel
        supported = NRMCSTable(mcs_table_index, mcs_category, min_coderate=MIN_LDPC_CODERATE, device=device).indices
        if mcs_index is not None and mcs_index not in supported:
            raise ValueError(f"MCS {mcs_index}: not one Sionna PHY can encode (code rates below 1/5 are not supported); "
                             f"one of {supported}")
        self.error_model = SionnaErrorModel(mcs_index=mcs_index, bler_target=bler_target, mcs_table_index=mcs_table_index,
                                            mcs_category=mcs_category, device=device, precision=precision,
                                            mcs_indices=supported)
        self.device = self.error_model.device
        self._qam = {}
        mapper = Mapper(self.qam)
        encoder = NRTBEncoder('PUSCH' if mcs_category == 0 else 'PDSCH', num_bp_iter, self.device, precision)
        self.precision = precision
        super().__init__([encoder, mapper, OFDM(), ApplyChannel(), OneTapEqualizer(), Demapper(mapper), NRTBDecoder(encoder)],
                         codec, adaptation=NRLinkAdaptation(self.error_model), report=('mcs', 'resource_elements', 'tb_size'))

    def qam(
        self,
        num_bits_per_symbol: int, # An MCS's modulation order
    ) -> SionnaModulation:
        "The QAM of a modulation order, with APP demapping (built once)."
        if num_bits_per_symbol not in self._qam:
            self._qam[num_bits_per_symbol] = SionnaModulation('qam', num_bits_per_symbol, device=self.device,
                                                              precision=self.precision, demapping_method='app')
        return self._qam[num_bits_per_symbol]

In [ ]:
show_doc(NRChain.qam)

### Tests

In [ ]:
assert [type(b).__name__ for b in UncodedChain().stages] == ['NoCoding', 'Mapper', 'SingleCarrier', 'ApplyChannel',
                                                              'OneTapEqualizer', 'Demapper', 'HardDecision']
assert UncodedChain().batch and not UncodedChain(batch=False).batch and not UncodedChain().needs_sinr and UncodedChain(modulation=BPSK()).bits_per_symbol == 1

In [ ]:
#| sionna
nr = NRChain(device='cpu')
assert nr.needs_sinr and [type(b).__name__ for b in nr.stages] == ['NRTBEncoder', 'Mapper', 'OFDM', 'ApplyChannel',
                                                                  'OneTapEqualizer', 'Demapper', 'NRTBDecoder']
assert nr.qam(4) is nr.qam(4) and nr.qam(6).bits_per_symbol == 6
assert min(nr.error_model.mcs.tolist()) == 3                            # code rates below 1/5: not encodable
try:
    NRChain(mcs_index=0); raise AssertionError('no error')
except ValueError as e:
    assert '1/5' in str(e)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()